In [4]:
import joblib
import pandas as pd
import numpy as np
import os

# Clone the repository only if it doesn't already exist
repo_path = 'Renewable-Energy-Output-Prediction'
if not os.path.exists(repo_path):
    !git clone https://github.com/Saher427/Renewable-Energy-Output-Prediction.git
else:
    print(f"Repository '{repo_path}' already exists. Skipping cloning.")

# Define the expected feature names based on the model's training data
expected_features = [
    'Start_Hour', 'End_Hour', 'Day_of_Year', 'Temperature_C', 'Humidity_Percent',
    'Precipitation_mm', 'WindSpeed_kmh', 'Source_Wind', 'Season_Spring', 'Season_Summer',
    'Season_Winter', 'Day_Name_Monday', 'Day_Name_Saturday', 'Day_Name_Sunday',
    'Day_Name_Thursday', 'Day_Name_Tuesday', 'Day_Name_Wednesday', 'Month_Name_August',
    'Month_Name_December', 'Month_Name_February', 'Month_Name_January', 'Month_Name_July',
    'Month_Name_June', 'Month_Name_March', 'Month_Name_May', 'Month_Name_November',
    'Month_Name_October', 'Month_Name_September', 'Rainfall_Flag_Yes', 'Year'
]

# Define X_new with the correct number of features (30) and column names
# This is a placeholder with random data. You NEED to replace this with your actual data for prediction.
# For categorical features (like Day_Name, Month_Name, Season, Source_Wind, Rainfall_Flag),
# ensure you use one-hot encoded values (0 or 1).
# For numerical features, provide their respective values.
# Example: To predict for a specific day, you would set relevant one-hot encoded columns to 1 and others to 0.
X_new = pd.DataFrame(np.random.rand(1, len(expected_features)), columns=expected_features)

# Load final tuned XGBoost model
model = joblib.load(
    f"{repo_path}/Data/ModelResults/tuned_xgboost_model.pkl"
)

# Make prediction
prediction = model.predict(X_new)

print("Prediction:", prediction)

Repository 'Renewable-Energy-Output-Prediction' already exists. Skipping cloning.
Prediction: [2151.1472]


In [5]:
def predict_energy(
    date,
    start_hour,
    end_hour,
    source,
    temperature,
    humidity,
    precipitation,
    wind_speed,
    rainfall
):

    # Convert date
    date = pd.to_datetime(date)

    # Create basic date features
    year = date.year
    day_of_year = date.dayofyear
    day_name = date.day_name()
    month_name = date.month_name()

    # Determine season
    if month_name in ["December", "January", "February"]:
        season = "Winter"
    elif month_name in ["March", "April", "May"]:
        season = "Spring"
    elif month_name in ["June", "July", "August"]:
        season = "Summer"
    else:
        season = "Fall"

    # Create one-row dataframe
    row = pd.DataFrame({
        "Start_Hour": [start_hour],
        "End_Hour": [end_hour],
        "Day_of_Year": [day_of_year],
        "Temperature_C": [temperature],
        "Humidity_Percent": [humidity],
        "Precipitation_mm": [precipitation],
        "WindSpeed_kmh": [wind_speed],
        "Source_Wind": [1 if source == "Wind" else 0],
        "Rainfall_Flag_Yes": [1 if rainfall == "Yes" else 0],
        "Year": [year]
    })

    # One-hot columns used during training
    day_columns = [
        "Monday", "Saturday", "Sunday",
        "Thursday", "Tuesday", "Wednesday"
    ]

    month_columns = [
        "August", "December", "February", "January",
        "July", "June", "March", "May",
        "November", "October", "September"
    ]

    season_columns = [
        "Spring", "Summer", "Winter"
    ]

    # Add one-hot encoded day columns
    for day in day_columns:
        row[f"Day_Name_{day}"] = int(day_name == day)

    # Add month columns
    for month in month_columns:
        row[f"Month_Name_{month}"] = int(month_name == month)

    # Add season columns
    for s in season_columns:
        row[f"Season_{s}"] = int(season == s)

    # Ensure exact feature order
    feature_order = [
        "Start_Hour",
        "End_Hour",
        "Day_of_Year",
        "Temperature_C",
        "Humidity_Percent",
        "Precipitation_mm",
        "WindSpeed_kmh",
        "Source_Wind",
        "Season_Spring",
        "Season_Summer",
        "Season_Winter",
        "Day_Name_Monday",
        "Day_Name_Saturday",
        "Day_Name_Sunday",
        "Day_Name_Thursday",
        "Day_Name_Tuesday",
        "Day_Name_Wednesday",
        "Month_Name_August",
        "Month_Name_December",
        "Month_Name_February",
        "Month_Name_January",
        "Month_Name_July",
        "Month_Name_June",
        "Month_Name_March",
        "Month_Name_May",
        "Month_Name_November",
        "Month_Name_October",
        "Month_Name_September",
        "Rainfall_Flag_Yes",
        "Year"
    ]

    row = row[feature_order]

    # Make prediction
    prediction = model.predict(row)[0]

    return prediction

In [6]:
prediction = predict_energy(
    date="2025-07-15",
    start_hour=14,
    end_hour=15,
    source="Wind",
    temperature=30,
    humidity=60,
    precipitation=0,
    wind_speed=10,
    rainfall="No"
)

print(f"Predicted Energy Output: {prediction:.2f} MWh")

Predicted Energy Output: 13705.33 MWh


In [7]:
prediction = predict_energy(
    date="2025-07-15",
    start_hour=13,
    end_hour=14,
    source="Solar",
    temperature=32,
    humidity=50,
    precipitation=0,
    wind_speed=6,
    rainfall="No"
)

print(f"Predicted Solar Output: {prediction:.2f} MWh")

Predicted Solar Output: 11002.45 MWh


In [8]:
prediction = predict_energy(
    date="2025-11-30",
    start_hour=21,
    end_hour=22,
    source="Wind",
    temperature=11.8,
    humidity=72,
    precipitation=0,
    wind_speed=5.9,
    rainfall="No"
)

actual = 5281

error = actual - prediction
percentage_error = abs(error) / actual * 100

print(f"Actual Production:     {actual:.2f} MWh")
print(f"Predicted Production:  {prediction:.2f} MWh")
print(f"Error:                 {error:.2f} MWh")
print(f"Percentage Error:      {percentage_error:.2f}%")

Actual Production:     5281.00 MWh
Predicted Production:  5022.92 MWh
Error:                 258.08 MWh
Percentage Error:      4.89%


In [9]:
import os
import glob

repo_path = "Renewable-Energy-Output-Prediction"

csv_files = glob.glob(
    os.path.join(repo_path, "**", "*.csv"),
    recursive=True
)

print("CSV files found:")
for f in csv_files:
    print(f)

CSV files found:
Renewable-Energy-Output-Prediction/Data/Raw/Energy Production Dataset.csv
Renewable-Energy-Output-Prediction/Data/Cleaned/Cleaned_Readable_Data.csv
Renewable-Energy-Output-Prediction/Data/Cleaned/Cleaned_Production_Data.csv
Renewable-Energy-Output-Prediction/Data/Modified Dataset/Dataset_with_Weather_features.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/tuned_xgboost_results.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/baseline_model_coefficients(without weather).csv
Renewable-Energy-Output-Prediction/Data/ModelResults/advanced_model_results.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/final_model_comparison.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/baseline_model_coefficients.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/baseline_model_results.csv
Renewable-Energy-Output-Prediction/Data/ModelResults/baseline_model_results(without weather).csv


In [10]:
# ==========================================
# AUTOMATIC TEST: 10 SOLAR + 10 WIND
# ==========================================

import pandas as pd
import numpy as np

# Load the dataset
data_path = "Renewable-Energy-Output-Prediction/Data/Modified Dataset/Dataset_with_Weather_features.csv"
df = pd.read_csv(data_path)

# Convert date column
df["Date"] = pd.to_datetime(df["Date"])

# Select 10 Solar and 10 Wind records
solar_test = df[df["Source"] == "Solar"].sample(10, random_state=42)
wind_test = df[df["Source"] == "Wind"].sample(10, random_state=42)

# Combine them
test_data = pd.concat([solar_test, wind_test]).sort_values(["Source", "Date", "Start_Hour"])

results = []

# Run prediction for every selected record
for _, row in test_data.iterrows():

    # Determine rainfall flag
    rainfall = "Yes" if row["Rainfall_Flag"] == "Yes" else "No"

    # Predict using your existing function
    predicted = predict_energy(
        date=row["Date"],
        start_hour=row["Start_Hour"],
        end_hour=row["End_Hour"],
        source=row["Source"],
        temperature=row["Temperature_C"],
        humidity=row["Humidity_Percent"],
        precipitation=row["Precipitation_mm"],
        wind_speed=row["WindSpeed_kmh"],
        rainfall=rainfall
    )

    actual = row["Production"]

    absolute_error = abs(actual - predicted)
    percentage_error = (absolute_error / actual) * 100

    results.append({
        "Date": row["Date"].date(),
        "Start_Hour": row["Start_Hour"],
        "End_Hour": row["End_Hour"],
        "Source": row["Source"],
        "Actual_MWh": actual,
        "Predicted_MWh": predicted,
        "Absolute_Error_MWh": absolute_error,
        "Percentage_Error": percentage_error
    })

# Create comparison table
comparison_table = pd.DataFrame(results)

# Display table
comparison_table

,Date,Start_Hour,End_Hour,Source,Actual_MWh,Predicted_MWh,Absolute_Error_MWh,Percentage_Error
0,2020-07-11,14,15,Solar,5898,5713.500000,184.500000,3.128179
1,2021-06-11,18,19,Solar,3490,3558.328613,68.328613,1.957840
2,2021-09-03,9,10,Solar,2217,2455.119141,238.119141,10.740602
3,2022-06-02,14,15,Solar,7032,7267.026367,235.026367,3.342241
4,2023-05-18,13,14,Solar,10403,9427.729492,975.270508,9.374896
5,2023-05-28,8,9,Solar,4487,4619.231445,132.231445,2.946990
6,2024-06-12,13,14,Solar,7608,8525.759766,917.759766,12.063088
7,2024-08-14,15,16,Solar,6313,6399.775391,86.775391,1.374551
8,2024-08-16,15,16,Solar,9223,8554.040039,668.959961,7.253171
9,2025-08-18,9,10,Solar,5542,5691.645996,149.645996,2.700217


In [11]:
# ==========================================
# SOLAR vs WIND PERFORMANCE SUMMARY
# ==========================================

summary = comparison_table.groupby("Source").agg(
    Records=("Source", "count"),
    MAE_MWh=("Absolute_Error_MWh", "mean"),
    RMSE_MWh=("Absolute_Error_MWh",
              lambda x: np.sqrt(np.mean(x**2))),
    Mean_Percentage_Error=("Percentage_Error", "mean"),
    Median_Percentage_Error=("Percentage_Error", "median")
).reset_index()

summary

,Source,Records,MAE_MWh,RMSE_MWh,Mean_Percentage_Error,Median_Percentage_Error
0,Solar,10,365.661713,493.862244,5.488177,3.235210
1,Wind,10,738.444824,1183.870483,14.981134,9.962679


In [12]:
# ==========================================
# FULL MODEL EVALUATION: SOLAR vs WIND
# ==========================================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the complete dataset
data_path = "Renewable-Energy-Output-Prediction/Data/Modified Dataset/Dataset_with_Weather_features.csv"
df = pd.read_csv(data_path)

df["Date"] = pd.to_datetime(df["Date"])

print("Total records:", len(df))
print("Solar records:", (df["Source"] == "Solar").sum())
print("Wind records:", (df["Source"] == "Wind").sum())


# ------------------------------------------
# Function to evaluate one energy source
# ------------------------------------------

def evaluate_source(source_name):

    source_df = df[df["Source"] == source_name].copy()

    actual_values = []
    predicted_values = []

    for _, row in source_df.iterrows():

        rainfall = "Yes" if row["Rainfall_Flag"] == "Yes" else "No"

        predicted = predict_energy(
            date=row["Date"],
            start_hour=row["Start_Hour"],
            end_hour=row["End_Hour"],
            source=row["Source"],
            temperature=row["Temperature_C"],
            humidity=row["Humidity_Percent"],
            precipitation=row["Precipitation_mm"],
            wind_speed=row["WindSpeed_kmh"],
            rainfall=rainfall
        )

        actual_values.append(row["Production"])
        predicted_values.append(predicted)

    actual_values = np.array(actual_values)
    predicted_values = np.array(predicted_values)

    absolute_errors = np.abs(actual_values - predicted_values)

    # Percentage error
    percentage_errors = (
        absolute_errors / np.abs(actual_values)
    ) * 100

    # Metrics
    mae = mean_absolute_error(actual_values, predicted_values)

    rmse = np.sqrt(
        mean_squared_error(actual_values, predicted_values)
    )

    r2 = r2_score(actual_values, predicted_values)

    mean_percentage_error = np.mean(percentage_errors)
    median_percentage_error = np.median(percentage_errors)

    return {
        "Source": source_name,
        "Records": len(source_df),
        "MAE_MWh": mae,
        "RMSE_MWh": rmse,
        "R2": r2,
        "Mean_Percentage_Error": mean_percentage_error,
        "Median_Percentage_Error": median_percentage_error
    }


# ------------------------------------------
# Evaluate Solar and Wind
# ------------------------------------------

solar_results = evaluate_source("Solar")
wind_results = evaluate_source("Wind")

# Create comparison table
full_comparison = pd.DataFrame([
    solar_results,
    wind_results
])

# Round values for easier reading
full_comparison = full_comparison.round({
    "MAE_MWh": 2,
    "RMSE_MWh": 2,
    "R2": 4,
    "Mean_Percentage_Error": 2,
    "Median_Percentage_Error": 2
})

print("\n========== FULL MODEL EVALUATION ==========\n")

full_comparison

Total records: 51862
Solar records: 9378
Wind records: 42484

========== FULL MODEL EVALUATION ==========



,Source,Records,MAE_MWh,RMSE_MWh,R2,Mean_Percentage_Error,Median_Percentage_Error
0,Solar,9378,350.61,547.76,0.9484,6.61,4.13
1,Wind,42484,719.66,1036.50,0.9403,20.68,9.37


## Conclusion

The full model evaluation reveals key differences in prediction performance between Solar and Wind energy sources:

- **Solar Energy Prediction:**
  - The model performs strongly for solar energy, with a Mean Absolute Error (MAE) of **350.61 MWh** and a Root Mean Squared Error (RMSE) of **547.76 MWh**.
  - The R-squared (R2) value of **0.9484** indicates that approximately 94.84% of the variance in solar energy production is explained by the model.
  - The Mean Percentage Error is **6.61%**, with a Median Percentage Error of **4.13%**, suggesting good overall accuracy and consistency for solar predictions.

- **Wind Energy Prediction:**
  - For wind energy, the model shows a higher Mean Absolute Error (MAE) of **719.66 MWh** and a RMSE of **1036.50 MWh**.
  - The R-squared (R2) value is **0.9403**, which is still quite good, but slightly lower than for solar, indicating a marginally less accurate fit for wind data.
  - The Mean Percentage Error for wind is significantly higher at **20.68%**, with a Median Percentage Error of **9.37%**. This suggests that while the model generally captures trends, there are larger individual deviations and more variability in prediction accuracy for wind energy compared to solar.

**Overall:**
The model provides robust predictions for both renewable energy sources, with slightly better performance and lower percentage errors for solar energy. The higher errors in wind prediction might be attributed to the inherent intermittency and variability of wind patterns, which can be more challenging to model accurately. Further improvements could potentially be explored by incorporating more detailed wind forecasting data or advanced time-series modeling techniques specific to wind dynamics.